In [ ]:
!pip install -q pycolmap torchmetrics gsplat
!pip install -q git+https://github.com/compphoto/Intrinsic.git
!pip install -q opencv-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
rm -rf /root/.cache/torch_extensions/*/gsplat*

In [ ]:
import os

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install -q git+https://github.com/compphoto/Intrinsic.git
    !pip install opencv-python
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
except ModuleNotFoundError:
    pass  # running locally, already in project root

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:

import argparse, glob, os
import numpy as np, torch
import torch.nn.functional as F
from PIL import Image
from gsplat import rasterization
from gsplat.strategy import DefaultStrategy
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
import os

dev = "cuda"

def get_image_paths(root):
  img_dir = os.path.join(root, "images_8")
  paths = []
  for path in os.listdir(img_dir):
    if (path.endswith(".JPG")):
      paths.append(os.path.join(img_dir, path))

  return paths

import pycolmap

def getw2c(image):
    cam_from_world = image.cam_from_world
    if callable(cam_from_world):
        cam_from_world = cam_from_world()

    w2c = np.eye(4)
    w2c[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float64)
    return w2c

def get_cameras(root, paths):
  colmap_dir = os.path.join(root, "sparse/0")

  reconstruction = pycolmap.Reconstruction(colmap_dir)
  cameras = _as_dict(reconstruction.cameras)

  w2c_mats = []
  camera_ids = []
  by_name = {}

  images = _as_dict(reconstruction.images)
  image_ids = [int(image_id) for image_id in reconstruction.reg_image_ids()]
  imdata = {image_id: images[image_id] for image_id in image_ids}


  for image_id in imdata:
    im = imdata[image_id]
    w2c_mats.append(getw2c(im))
    camera_ids.append(im.camera_id)
    by_name[im.name] = (getw2c(im), int(im.camera_id))


  width, height = Image.open(paths[0]).size
  viewmats = []
  intrinsics = []
  for path in paths:
    name = os.path.basename(path)
    w2c, camera_id = by_name[name]
    cam = cameras[camera_id]
    K = np.asarray(cam.calibration_matrix(), dtype=np.float64).copy()
    K[0, :] *= width / cam.width
    K[1, :] *= height / cam.height

    intrinsics.append(K)
    viewmats.append(w2c)

  points = np.array([p.xyz for p in reconstruction.points3D.values()], dtype=np.float32)

  return (np.stack(viewmats), np.stack(intrinsics), (width, height), points,)


In [ ]:
def _as_dict(map_like):
    return {int(k): v for k, v in map_like.items()}

In [ ]:
import pycolmap

ROOT = "/content/drive/MyDrive/3dgs/data/mipnerf360/room"

paths = get_image_paths(ROOT)

viewmats, intrinsics, (width, height), points = get_cameras(ROOT, paths)

In [ ]:
def get_views(paths, root, predictor, scene, dataset):

  images = []
  albedos = []
  depths = []

  for p in paths:
    file_name = os.path.basename(p).split('.')[0]
    print(file_name)
    i = torch.tensor(np.array(Image.open(p).convert("RGB")), dtype=torch.float32) / 255
    width, height = Image.open(p).size


    assert i.ndim == 3 and i.shape[2] == 3, (
            f"Expected (H, W, 3) for {p}, got {tuple(i.shape)}"
        )


    albedo_path = os.path.join(root, predictor, scene, dataset, "albedos", f"{file_name}.npy")
    albedo = torch.tensor(np.load(albedo_path), dtype=torch.float32)
    albedos.append(albedo)

    depth_path = os.path.join(root, "depth_anything_v2", "depths", scene,
                              dataset, f"{file_name}.png",
  )

    depth = torch.tensor(
        np.array(Image.open(depth_path), dtype=np.float32), dtype=torch.float32
    )

    depth_npy_path = os.path.splitext(depth_path)[0] + ".npy"
    np.save(depth_npy_path, depth.detach().cpu().numpy())

    depths.append(depth.detach().clone())
    images.append(i)

  return torch.stack(images).to(dev), torch.stack(albedos).to(dev), torch.stack(depths).to(dev)


In [ ]:
# load cameras

path = get_image_paths('data/mipnerf360/room')

images, albedos, depths = get_views(path[0:2], "/content/drive/MyDrive/3dgs", "rgbx", "mipnerf360", "room")
# TESTED HERE


DSCF4667
DSCF4668


In [ ]:
def pearson_loss(d, m):
    d, m = d.flatten(), m.flatten()
    d = (d - d.mean()) / (d.std() + 1e-6); m = (m - m.mean()) / (m.std() + 1e-6)
    return 1 - (d * m).mean()

def photo_loss(pred, tgt, lam=0.2):
    l1 = (pred - tgt).abs().mean()
    s = 1 - ssim_fn(pred.permute(2, 0, 1)[None], tgt.permute(2, 0, 1)[None])
    return (1 - lam) * l1 + lam * s

def tv(x):
    return (x[1:] - x[:-1]).abs().mean() + (x[:, 1:] - x[:, :-1]).abs().mean()

In [ ]:

learning_rates = {
    "means": 1.6e-4,
    "scales": 5e-3,
    "quats": 1e-3,
    "opacities": 5e-2,
    "colors": 1e-2,
}

def create_gaussians(means, scales, quats, opacities, colors, scene_scale=1.0):
    params = torch.nn.ParameterDict({
        "means": torch.nn.Parameter(means),
        "scales": torch.nn.Parameter(scales),
        "quats": torch.nn.Parameter(quats),
        "opacities": torch.nn.Parameter(opacities),
        "colors": torch.nn.Parameter(colors),
    })
    optimizers = {
        "means": torch.optim.Adam([params["means"]], lr=learning_rates["means"] * scene_scale),
        "scales": torch.optim.Adam([params["scales"]], lr=learning_rates["scales"]),
        "quats": torch.optim.Adam([params["quats"]], lr=learning_rates["quats"]),
        "opacities": torch.optim.Adam([params["opacities"]], lr=learning_rates["opacities"]),
        "colors": torch.optim.Adam([params["colors"]], lr=learning_rates["colors"]),
    }
    return params, optimizers

def initialise_gaussians(points, init_scale=1.0, init_opacity=0.1, scene_scale=1.0, dev=dev):
    pts = torch.tensor(points, dtype=torch.float32, device=dev)
    num_pts = len(pts)

    distances = torch.cdist(pts, pts)
    nearest_distances = distances.topk(4, largest=False).values
    average_distance = nearest_distances[:, 1:].mean(dim=-1).clamp(min=1e-4)

    scales = torch.log(average_distance * init_scale).unsqueeze(-1).repeat(1, 3)
    quats = F.normalize(torch.rand((num_pts, 4), dtype=torch.float32, device=dev), dim=-1)
    opacities = torch.logit(torch.full((num_pts,), init_opacity, dtype=torch.float32, device=dev))
    colors = torch.zeros((num_pts, 3), dtype=torch.float32, device=dev)

    return create_gaussians(pts, scales, quats, opacities, colors, scene_scale=scene_scale)



In [ ]:
def initialise_from_albedos(checkpoint, color_shape, initial_color=0.0):
  means = checkpoint["means"].to(dev).clone()
  scales = checkpoint["scales"].to(dev).clone()
  quats = checkpoint["quats"].to(dev).clone()
  opacities = checkpoint["opacities"].to(dev).clone()

  num_gaussians = len(means)
  colors = torch.full((num_gaussians, *color_shape), initial_color, device=dev)

  return create_gaussians(means, scales, quats, opacities, colors)


In [ ]:
from gsplat import rasterization

def render_gaussians(gaussians, viewmat, K, width, height, render_mode="RGB",sh_degree=None,
    color_activation=None):
    means = gaussians["means"]
    quats = gaussians["quats"]
    scales = torch.exp(gaussians["scales"])
    opacities = torch.sigmoid(gaussians["opacities"])
    raw_colors = gaussians["colors"]

    if color_activation is not None:
        colors = color_activation(raw_colors)
    else:
        colors = raw_colors

    rendered_colors, rendered_alphas, info = rasterization(
        means=means,
        quats=quats,
        scales=scales,
        opacities=opacities,
        colors=colors,
        viewmats=viewmat.unsqueeze(0),
        Ks=K.unsqueeze(0),
        width=width,
        height=height,
        sh_degree=sh_degree,
        render_mode=render_mode,
    )

    return rendered_colors[0], rendered_alphas[0], info

albdo = torch.sigmoid
shdo = F.softplus
resdo = lambda c: c

In [ ]:
def step(loss, optimizers, strategy=None, state=None, params=None, step_idx=0, info=None):
    loss.backward()
    if strategy is not None:
        strategy.step_post_backward(
            params=params,
            optimizers=optimizers,
            state=state,
            step=step_idx,
            info=info,
        )
    for optimizer in optimizers.values():
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)


def save_checkpoint(params, checkpoint_directory, scene, predictor):
    os.makedirs(checkpoint_directory, exist_ok=True)
    path = os.path.join(checkpoint_directory, f"{scene}_{predictor}.pt")
    checkpoint = {name: tensor.detach().cpu() for name, tensor in params.items()}
    torch.save(checkpoint, path)
    print("Saved to: " + path)